# Task 2: Predicting Rainfall
## Singapore Rainfall, 2017–2024

## 2.1 Spark Session

In [ ]:
import socket
from functools import reduce

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

from pyspark.sql import SparkSession, Window, DataFrame

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pyspark.sql.functions as F
import seaborn as sns
import pandas as pd

import os
import sys

import time
from pyspark import StorageLevel
from pyspark.ml.functions import vector_to_array

In [ ]:
# WHY REPLACE:
# Your logs showed stages with thousands of tasks.
# Start with a modest shuffle partition count to reduce scheduling overhead.
# These settings are starting points, not guaranteed optimal cluster settings.

spark = SparkSession.builder.getOrCreate()

spark.conf.set("spark.sql.session.timeZone", "Asia/Singapore")
spark.conf.set("spark.sql.shuffle.partitions", "128")
spark.conf.set("spark.sql.adaptive.enabled", "true")

print("Running on:   ", socket.gethostname())
print("Spark master: ", spark.sparkContext.master)
print("Spark version:", spark.version)

Running on:    rainfall-dev-m
Spark master:  yarn
Spark version: 3.5.3


In [3]:
# Rainfall Multiplier per Day there are 12 five-minute intervals per hour and 24 hours per day

RF_MULTIPLIER = (24 * 60) / 5

## 2.2 Reading the Cleaned Rainfall Data

In [4]:
rain = spark.read.parquet("gs://dsa5208-rainfall-sixing-data/parquet/")

In [5]:
# Step 1: Extract the year and month from the observation date
rain = (
    rain.withColumn("date", F.to_date("date"))
        .withColumn("year", F.year("date"))
        .withColumn("month", F.month("date"))
)

# Step 2: Calculate mean rainfall per observed five-minute interval
# for each year and month
monthly_spk = (
    rain.filter(F.col("year").between(2017, 2024))
        .groupBy("year", "month")
        .agg(
            F.avg("reading_value").alias("mean_per_interval"),
            F.countDistinct("station_id").alias("n_stations")
        )
        .withColumn(
            "year_month",
            F.make_date("year", "month", F.lit(1))
        )
        .withColumn(
            "days",
            F.dayofmonth(F.last_day("year_month"))
        )
        # mean interval rainfall × expected intervals in the month
        .withColumn(
            "avg_rainfall_monthly",
            F.col("mean_per_interval") * F.col("days") * RF_MULTIPLIER
        )
        .orderBy("year_month")
)

## 2.3 Auto-Correlation of Rainfall and Its Timelags 

In [ ]:
def lookback(seconds):
    return (Window.partitionBy("station_id")
            .orderBy("ts_long")
            .rangeBetween(-seconds, 0))

In [7]:
rain_lag = rain.withColumn("ts_long", F.col("timestamp").cast("long"))

w = Window.partitionBy("station_id").orderBy("ts_long")

lags = {"lag_5m": 1, "lag_30m": 6, "lag_1h": 12}

for name, k in lags.items():
    rain_lag = (rain_lag
        .withColumn(name, F.lag("reading_value", k).over(w))
        .withColumn(name + "_ts", F.lag("ts_long", k).over(w)))
    
    # null out where the gap isn't the expected duration
    rain_lag = rain_lag.withColumn(name,
        F.when(F.col("ts_long") - F.col(name + "_ts") == k * 300,
               F.col(name)))
    rain_lag = rain_lag.drop(name + "_ts")

In [ ]:
# WHY REPLACE:
# 1. The original 30-minute window included 7 readings instead of 6.
# 2. The original 60-minute window included 13 readings instead of 12.
# 3. A future sum can be non-null even when readings are missing.
#    We must check counts and time gaps before using it as a label.

# Spark includes both endpoints of these windows.
past_30m = lookback(1500)     # Six readings: t−25 through t.
past_1h = lookback(3300)      # Twelve readings: t−55 through t.
history_1h = lookback(3600)   # Thirteen readings, including t−60 for lag_1h.

future_window = (
    Window.partitionBy("station_id")
          .orderBy("ts_long")
          .rangeBetween(1, 1800)
)

# Flag gaps that are not exactly five minutes.
rain_lag = (
    rain_lag
    .withColumn("_previous_ts", F.lag("ts_long").over(w))
    .withColumn(
        "_bad_gap",
        F.when(
            F.col("ts_long") - F.col("_previous_ts") == 300,
            0
        ).otherwise(1)
    )
)

rain_lag = (
    rain_lag

    # Historical rainfall features.
    .withColumn(
        "rain_cumsum_30m",
        F.sum("reading_value").over(past_30m)
    )
    .withColumn(
        "rain_cumsum_1h",
        F.sum("reading_value").over(past_1h)
    )
    .withColumn(
        "wet_count_30m",
        F.sum(
            (F.col("reading_value") > 0).cast("long")
        ).over(past_30m)
    )

    # Check the thirteen historical readings and twelve connecting gaps.
    .withColumn(
        "_history_count",
        F.count("reading_value").over(history_1h)
    )
    .withColumn(
        "_history_bad_gap",
        F.max("_bad_gap").over(past_1h)
    )

    # Check the six future readings and their connecting gaps.
    .withColumn(
        "_future_count",
        F.count("reading_value").over(future_window)
    )
    .withColumn(
        "_future_bad_gap",
        F.max("_bad_gap").over(future_window)
    )
    .withColumn(
        "future_30m",
        F.sum("reading_value").over(future_window)
    )

    # Use the observation timestamp consistently.
    .withColumn("hour", F.hour("timestamp"))
    .withColumn("month", F.month("timestamp"))
    .withColumn("year", F.year("timestamp"))
)

In [ ]:
# WHY REPLACE:
# 1. Require complete windows; non-null future rainfall alone is insufficient.
# 2. Cache the feature table so later actions can reuse the window calculations.
# 3. Remove the five-way union; threshold expansion happens in cell 10.
# 4. Prevent training labels from using rainfall beyond the training period.

feature_cols = [
    "location_latitude",
    "location_longitude",
    "hour",
    "month",
    "reading_value",
    "lag_5m",
    "lag_30m",
    "lag_1h",
    "rain_cumsum_30m",
    "rain_cumsum_1h",
    "wet_count_30m",
    "thr"
]

THRESHOLDS = [0.0, 1.0, 2.5, 5.0, 10.0]
past_feature_cols = [c for c in feature_cols if c != "thr"]

base = (
    rain_lag
    .filter(
        (F.col("_history_count") == 13) &
        (F.col("_history_bad_gap") == 0) &
        (F.col("_future_count") == 6) &
        (F.col("_future_bad_gap") == 0)
    )
    .filter(F.col("year").between(2017, 2024))
    .select(
        "station_id", "timestamp", "year",
        *past_feature_cols,
        "future_30m"
    )
    .dropna(subset=past_feature_cols + ["future_30m"])

    # Store on executor disks instead of trying to hold everything in memory.
    .persist(StorageLevel.DISK_ONLY)
)

# Complete feature preparation now, so it is not included in model-fit timing.
# This first full-data preparation step can still take time.
started = time.perf_counter()

print("Eligible station-time rows:", base.count())
print(
    "Feature preparation seconds:",
    round(time.perf_counter() - started, 1)
)

target_end = F.col("timestamp") + F.expr("INTERVAL 30 MINUTES")

train = base.filter(
    (F.col("year") <= 2022) &
    (target_end < F.lit("2023-01-01").cast("timestamp"))
)

val = base.filter(
    (F.col("year") == 2023) &
    (target_end < F.lit("2024-01-01").cast("timestamp"))
)

test = base.filter(
    (F.col("year") == 2024) &
    (target_end < F.lit("2025-01-01").cast("timestamp"))
)

+-------------------+
|      positive_rate|
+-------------------+
|0.02179559928466956|
+-------------------+



In [ ]:
# WHY REPLACE:
# The original code kept all positive examples but only 5% of negatives.
# That made exceedances appear much more common in training.
#
# This version:
# 1. Takes a small uniform sample to reduce training time.
# 2. Gives positive and negative examples the same selection chance.
# 3. Prints their counts separately for every rainfall threshold.
#
# It does NOT force the data to become 50% positive and 50% negative.

SEED = 42

# These are starting fractions for checking that training runs successfully.
# Increase them later if too few positive examples are retained.
TRAIN_FRACTION = 0.005       # Approximately 0.5% of training station-times.
VALIDATION_FRACTION = 0.05   # Approximately 5% of validation station-times.

ML_PARTITIONS = 32


def prepare_sample(df, fraction):

    # STEP 1: Randomly select station-time observations.
    # Selection does not depend on whether rainfall exceeds a threshold.
    #
    # Historical features and future rainfall were already calculated.
    # Do not sample raw readings before calculating those windows.
    sampled = df.sample(
        withReplacement=False,
        fraction=fraction,
        seed=SEED
    )

    # STEP 2: Ask all five threshold questions for each selected observation.
    #
    # Example: if future_30m = 3 mm:
    # thr = 1 mm -> label 1
    # thr = 5 mm -> label 0
    labelled = (
        sampled
        .withColumn(
            "thr",
            F.explode(
                F.array(*[F.lit(float(t)) for t in THRESHOLDS])
            )
        )
        .withColumn(
            "label",
            (F.col("future_30m") > F.col("thr")).cast("double")
        )
    )

    # STEP 3: Keep only model inputs and the label.
    # Cache this smaller dataset so repeated training actions can reuse it.
    return (
        labelled
        .select(*feature_cols, "label")
        .repartition(ML_PARTITIONS)
        .persist(StorageLevel.MEMORY_AND_DISK)
    )


train_final = prepare_sample(train, TRAIN_FRACTION)
val_final = prepare_sample(val, VALIDATION_FRACTION)


# STEP 4: Inspect imbalance separately for each threshold.
# A higher rainfall threshold may have fewer positive examples.
for dataset_name, dataset in [
    ("Training", train_final),
    ("Validation", val_final)
]:
    summary = (
        dataset
        .groupBy("thr")
        .agg(
            F.count("*").alias("total"),
            F.sum("label").alias("positives")
        )
        .orderBy("thr")
        .collect()
    )

    # This action also fills the cache before model training starts.
    print(f"\n{dataset_name} class distribution")

    if not summary:
        raise ValueError(f"{dataset_name} sample is empty.")

    for row in summary:
        total = row["total"]
        positives = int(row["positives"])
        negatives = total - positives
        positive_pct = 100 * positives / total

        print(
            f"> {row['thr']:g} mm: "
            f"label 1 = {positives:,}, "
            f"label 0 = {negatives:,}, "
            f"positive percentage = {positive_pct:.2f}%"
        )

        # Stop if a threshold has no examples of one class.
        # Increase the relevant sample fraction and rerun this cell.
        if positives == 0 or negatives == 0:
            raise ValueError(
                f"{dataset_name}, threshold {row['thr']:g} mm: "
                "both classes are required. Increase the sample fraction."
            )

# IMPORTANT:
# Having both classes is only a minimum check.
# A handful of positive examples is not enough for reliable modelling.
#
# Keep validation and final test data at their natural class proportions.
# Do not balance them by selectively removing negative examples.

26/10/14 13:24:33 WARN SimpleTableFunctionRegistry: The function read_files replaced a previously registered function.
26/10/14 13:24:33 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/10/14 13:24:33 WARN SimpleTableFunctionRegistry: The function read_files replaced a previously registered function.
26/10/14 13:24:34 WARN SimpleTableFunctionRegistry: The function read_files replaced a previously registered function.


+------------------+
|     positive_rate|
+------------------+
|0.3082570415959772|
+------------------+



+----+------+
| thr| n_pos|
+----+------+
| 0.0|201767|
| 1.0| 81485|
| 2.5| 46192|
| 5.0| 26482|
|10.0| 12074|
+----+------+



In [ ]:
#train_final.write.mode("overwrite").parquet("gs://dsa5208-rainfall-sixing-data/train_parquet/")
#val_final.write.mode("overwrite").parquet("gs://dsa5208-rainfall-sixing-data/val_parquet/")
#test_final.write.mode("overwrite").parquet("gs://dsa5208-rainfall-sixing-data/test_parquet/")

In [12]:
assembler = VectorAssembler(inputCols=feature_cols,
                            outputCol="features",
                            handleInvalid="skip")

gbt = GBTClassifier(featuresCol="features", labelCol="label",
                    maxDepth=5, maxIter=50, stepSize=0.1, seed=42)

train_v = assembler.transform(train_final)
test_v  = assembler.transform(val_final)

model = gbt.fit(train_v)
preds = model.transform(test_v)

pr  = BinaryClassificationEvaluator(metricName="areaUnderPR",
        labelCol="label", rawPredictionCol="rawPrediction").evaluate(preds)
roc = BinaryClassificationEvaluator(metricName="areaUnderROC",
        labelCol="label", rawPredictionCol="rawPrediction").evaluate(preds)

base_rate = preds.select(F.avg("label")).first()[0]
print(f"PR-AUC  {pr:.4f}   (baseline {base_rate:.4f})")
print(f"ROC-AUC {roc:.4f}")

26/10/14 13:46:27 ERROR AsyncEventQueue: Dropping event from queue eventLog. This likely means one of the listeners is too slow and cannot keep up with the rate at which tasks are being started by the scheduler.
26/10/14 13:46:27 WARN AsyncEventQueue: Dropped 1 events from eventLog since the application started.
26/10/14 13:47:27 WARN AsyncEventQueue: Dropped 85107 events from eventLog since Wed Oct 14 13:46:27 UTC 2026.
26/10/14 13:48:27 WARN AsyncEventQueue: Dropped 88289 events from eventLog since Wed Oct 14 13:47:27 UTC 2026.
26/10/14 13:49:27 WARN AsyncEventQueue: Dropped 86128 events from eventLog since Wed Oct 14 13:48:27 UTC 2026.
26/10/14 13:50:27 WARN AsyncEventQueue: Dropped 87939 events from eventLog since Wed Oct 14 13:49:27 UTC 2026.
26/10/14 13:51:27 WARN AsyncEventQueue: Dropped 90869 events from eventLog since Wed Oct 14 13:50:27 UTC 2026.
26/10/14 13:52:27 WARN AsyncEventQueue: Dropped 92180 events from eventLog since Wed Oct 14 13:51:27 UTC 2026.
26/10/14 13:53:27 WA

KeyboardInterrupt: 

26/10/14 14:22:39 WARN AsyncEventQueue: Dropped 76306 events from eventLog since Wed Oct 14 14:21:39 UTC 2026.
26/10/14 14:23:39 WARN AsyncEventQueue: Dropped 74645 events from eventLog since Wed Oct 14 14:22:39 UTC 2026.
26/10/14 14:24:39 WARN AsyncEventQueue: Dropped 75262 events from eventLog since Wed Oct 14 14:23:39 UTC 2026.
26/10/14 14:25:39 WARN AsyncEventQueue: Dropped 74677 events from eventLog since Wed Oct 14 14:24:39 UTC 2026.
26/10/14 14:26:39 WARN AsyncEventQueue: Dropped 72144 events from eventLog since Wed Oct 14 14:25:39 UTC 2026.
26/10/14 14:27:39 WARN AsyncEventQueue: Dropped 78436 events from eventLog since Wed Oct 14 14:26:39 UTC 2026.
26/10/14 14:28:39 WARN AsyncEventQueue: Dropped 71560 events from eventLog since Wed Oct 14 14:27:39 UTC 2026.


In [ ]:
# feature importances
for name, imp in sorted(zip(feature_cols, model.featureImportances.toArray()),
                        key=lambda x: -x[1]):
    print(f"{imp:.4f}  {name}")

# persistence baseline 
naive = preds.withColumn("naive", (F.col("rain_cumsum_30m") > 0.5).cast("int"))
naive.groupBy("naive", "label").count().orderBy("naive", "label").show()

0.4837  rain_cumsum_30m
0.3491  reading_value
0.0506  hour
0.0499  rain_cumsum_1h
0.0244  month
0.0229  lag_5m
0.0130  lag_30m
0.0062  lag_1h
0.0001  location_longitude
0.0000  location_latitude


Row(avg(CAST((naive = label) AS INT))=0.9670901466257842)


In [18]:
naive = preds.withColumn("naive", (F.col("rain_cumsum_30m") > 0.5).cast("int"))

naive.groupBy("naive", "label").count().orderBy("naive", "label").show()

+-----+-----+-------+
|naive|label|  count|
+-----+-----+-------+
|    0|    0|6006697|
|    0|    1|  93060|
|    1|    0| 116011|
|    1|    1| 137071|
+-----+-----+-------+



In [19]:
preds.groupBy("prediction", "label").count().orderBy("prediction", "label").show()

+----------+-----+-------+
|prediction|label|  count|
+----------+-----+-------+
|       0.0|    0|6097169|
|       0.0|    1| 121179|
|       1.0|    0|  25539|
|       1.0|    1| 108952|
+----------+-----+-------+



In [21]:
base_rate = train.select(F.avg("label")).first()[0]

shuffled = train.withColumn("label",
    F.col("label") * 0 + (F.rand(seed=7) < base_rate).cast("int"))

m2 = gbt.fit(assembler.transform(shuffled))
p2 = m2.transform(assembler.transform(test))
print(BinaryClassificationEvaluator(metricName="areaUnderPR",
        labelCol="label", rawPredictionCol="rawPrediction").evaluate(p2))

0.1974824343394064


In [22]:
print(BinaryClassificationEvaluator(metricName="areaUnderROC",
        labelCol="label", rawPredictionCol="rawPrediction").evaluate(p2))

0.5637655553184941


26/10/11 16:21:28 ERROR TransportResponseHandler: Still have 1 requests outstanding when connection from /10.148.0.33:45252 is closed
26/10/11 16:21:28 ERROR TransportClient: Failed to send RPC RPC 8693474043611835936 to /10.148.0.33:45252: io.netty.channel.StacklessClosedChannelException
io.netty.channel.StacklessClosedChannelException: null
	at io.netty.channel.AbstractChannel$AbstractUnsafe.write(Object, ChannelPromise)(Unknown Source) ~[netty-transport-4.1.100.Final.jar:4.1.100.Final]
26/10/11 16:21:28 WARN BlockManagerMasterEndpoint: Error trying to remove RDD 2211 from block manager BlockManagerId(31, rainfall-dev-w-0.asia-southeast1-b.c.dsa5208-rainfall-sixing.internal, 40695, None)
java.io.IOException: Connection from /10.148.0.33:45252 closed
	at org.apache.spark.network.client.TransportResponseHandler.channelInactive(TransportResponseHandler.java:147) ~[spark-network-common_2.12-3.5.3.jar:3.5.3]
	at org.apache.spark.network.server.TransportChannelHandler.channelInactive(Trans